# 리포트 76 — 기울기 판정의 문턱은 세션간 진폭 재현성이고, σ 사슬 세대를 바꾸면 그 문턱이 손닿는 범위 밖으로 좁아진다

> ### 한 일
> **우리 커널의 밴드 기울기와 앵커 두 개의 기울기가 대역 끝에서 벌리는 간격을 계산해 세션 재현성 요구치로 못 박고, 세대를 바꿔 같은 계산을 다시 했다.**

### 결과
1. 판정 문턱은 1.77 dB [^1] 다 — 앵커 두 개(전대역 Das · 창을 맞춘 저대역 Yuan) 중 **좁은 쪽**이다.
2. 우리 커널은 0.936 [^2] ~ 1.517 [^3] dB/GHz 이고, 앵커는 전대역 0.210 [^4] · 창을 맞춘 쪽 0.411 [^5] dB/GHz 다.
3. 대역 3.367 GHz [^6] 를 지나며 두 가설이 벌리는 폭은 전대역 앵커에서 2.44 [^7] ~ 4.40 [^8] dB, 창을 맞춘 앵커에서 1.77 [^9] ~ 3.73 [^10] dB 다.
4. ⚠⚠ 이 문턱은 생산 원장 세대(2026-07-30 07:16:08 [^11])의 수다. 디스크의 현재 세대(2026-08-03 05:49:19 [^12])로 같은 정의를 다시 적합하면 Matrice 4E 가 0.936 [^2] → 0.233 dB/GHz [^13] 로 내려간다.
5. 그 세대에서 가장 좁은 판별폭은 0.08 dB [^14] 로, 세션 진폭 재현성이 닿는 범위 밖이다 — 이 행을 «결판» 으로 유지하려면 σ 사슬 재실행이 선행 조건이다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 기울기 정의 | 세 밴드 방위평균 μ 를 f[GHz] 에 1차 적합(el=0). 정의는 하나로 고정한다 — 세 밴드 방위평균 μ 를 f[GHz] 에 1차 적합(el=0) [^15] |
| 판별폭 | gap = (ours − anchor) × span — 대역 끝에서 두 가설이 벌리는 간격 [dB]. 세션간 진폭 재현성이 이보다 좋아야 기울기 판정이 성립한다 |
| 앵커 두 개 | 전대역 적합값(Das)과 이 캠페인에 창을 맞춘 저대역 적합값(Yuan θ=90 복원 실측곡선)을 나란히 쓴다. 문턱은 두 앵커가 주는 판별폭 중 **좁은 쪽**을 세션 재현성 요구로 쓴다 [^16] |
| 두 세대 | 생산 원장은 한 세대 앞선 `rcs_anchor.json` 위에 서 있다. 디스크의 현재 판으로 같은 정의를 다시 적합한 값을 나란히 싣는다 — 문턱은 생산 원장 값으로 그대로 둔다 |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/plan_measurement.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python -c "import sigma_anchor as S; S.write_measurement_plan()"
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report06_measurement.py
```

| | |
|---|---|
| 출력 | `outputs/report06_measurement.json`, `outputs/measurement_plan.json`, `outputs/report06_derived.json` |
| 소요 | 약 10 초 (CPU) |
| 비고 | 산문판 설계서는 `docs/MEASUREMENT_PLAN.md` 이고, 그 안의 수치표는 `src/sigma_anchor.py:939` 가 자동 주입한다 |

---

## 세션 재현성이 판정의 문턱이다

![report06_slope](../outputs/figures/report06_slope.png)

**그림 1.** 우리 기울기와 앵커 기울기를 가르려면 세션 재현성이 얼마나 좋아야 하는가?
우리 커널은 0.936 [^2] ~ 1.517 [^3] dB/GHz 다. 앵커는 **두 개를 나란히** 쓴다.

## 두 앵커와 두 창

| 앵커 | 기울기 [dB/GHz] | 적합 창 [GHz] |
|---|---|---|
| 전대역 (Das) | 0.210 [^4] | 1.8 [^17] ~ 18.2 [^18] |
| 창을 맞춘 저대역 (Yuan theta=90 복원 실측곡선 (EuCAP 2025) [^19]) | 0.411 [^5] | 1.8 [^20] ~ 6.0 [^21] |
| 이 캠페인의 창 | — | 1.843 [^22] ~ 5.21 [^23] |

⚠ 두 창은 **같지 않고 겹친다** — 앵커 창이 이 캠페인의 창을 덮되 위쪽이 더 넓다. 전대역 앵커와 견주면 훨씬 가깝다는 뜻이지 같은 창이라는 뜻이 아니다.

## 판별폭과 문턱

대역 3.367 GHz [^6] 를 지나며 두 가설이 벌리는 폭은 전대역 앵커에서 2.44 [^7] ~ 4.40 [^8] dB, 창을 맞춘 앵커에서 1.77 [^9] ~ 3.73 [^10] dB 다.

**판정 문턱은 둘 중 좁은 쪽 1.77 dB [^1] 로 잡는다** — 창을 맞춘 앵커가 우리 값에 더 가까워서 요구 재현성이 그만큼 빡빡하다([^24]).

## 세대를 바꾸면 문턱이 손닿는 범위 밖으로 간다

위의 «우리 커널» 값은 생산 원장(`sigma_anchor.json`, 2026-07-30 07:16:08 [^11] 판 `rcs_anchor.json` 위)에서 왔다. 디스크의 현재 `rcs_anchor.json`(2026-08-03 05:49:19 [^12] 판)으로 같은 정의를 다시 적합하면 Matrice 4E 가 0.936 [^2] → 0.233 dB/GHz [^13] 로 내려가고, 전대역 앵커와의 판별폭이 2.44 [^25] → 0.08 dB [^26] 가 된다.

그 세대에서 가장 좁은 판별폭은 0.08 dB [^14] 로, 세션 진폭 재현성이 닿는 범위 밖이다. 두 세대 모두 2026-08-04 [^27] 형상 정정 전 메쉬 위·2026-08-07 10:58:22 [^28] Γ(θ) 각도 모양(기본 켬) 이전 커널 위에 서 있고, 앵커 5기체 중 Matrice 4E 가 그 정정을 받았다 — 재실행 선행조건은 형상 정정 + Γ(θ) 다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| σ 사슬(rcs_anchor → sigma_anchor)을 형상 정정 + Γ(θ) 한 세대로 다시 돌린 뒤 이 문턱을 다시 낸다 | 기울기 판별폭이 현재 기하 위에서 확정된다 — 지금은 세대를 바꾸는 것만으로 2.44 [^25] → 0.08 dB [^26] 움직인다 | `benchmark/rcs_anchor.py` → `src/sigma_anchor.py` |
| 세 밴드를 같은 세션에서 재고 세션간 진폭 재현성을 기록한다 | 밴드 기울기가 우리 커널 값과 앵커 0.210 [^4] dB/GHz 중 어디에 앉는지 결정된다 — 위 사슬 재실행이 이 판정의 선행 조건이다 | [편 24 «앵커 모드»](24_anchor-mode.ipynb) 재기술 |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 28개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report06_derived.json` | `slope.threshold_db` | 1.768 |
| [^2] | `outputs/report06_derived.json` | `slope.rows[0].ours_db_per_ghz` | 0.9359 |
| [^3] | `outputs/report06_derived.json` | `slope.rows[1].ours_db_per_ghz` | 1.517 |
| [^4] | `outputs/report06_derived.json` | `slope.anchor_db_per_ghz` | 0.21 |
| [^5] | `outputs/report06_derived.json` | `slope.anchor_band_matched_db_per_ghz` | 0.4107 |
| [^6] | `outputs/report06_derived.json` | `slope.span_ghz` | 3.367 |
| [^7] | `outputs/report06_derived.json` | `slope.gap_db_min` | 2.444 |
| [^8] | `outputs/report06_derived.json` | `slope.rows[1].gap_db` | 4.401 |
| [^9] | `outputs/report06_derived.json` | `slope.gap_db_min_band_matched` | 1.768 |
| [^10] | `outputs/report06_derived.json` | `slope.rows[1].gap_db_band_matched` | 3.725 |
| [^11] | `outputs/report06_derived.json` | `slope.ledger_generation` | 2026-07-30 07:16:08 |
| [^12] | `outputs/report06_derived.json` | `slope.current_generation` | 2026-08-03 05:49:19 |
| [^13] | `outputs/report06_derived.json` | `slope.rows[0].ours_current_generation_db_per_ghz` | 0.2332 |
| [^14] | `outputs/report06_derived.json` | `slope.discrimination_min_abs_db_current_generation` | 0.07819 |
| [^15] | `outputs/report06_derived.json` | `slope.fit_note_short` | 세 밴드 방위평균 μ 를 f[GHz] 에 1차 적합(el=0) |
| [^16] | `outputs/report06_derived.json` | `slope.threshold_rule` | 두 앵커가 주는 판별폭 중 **좁은 쪽**을 세션 재현성 요구로 쓴다 |
| [^17] | `outputs/report06_derived.json` | `slope.anchor_window_ghz[0]` | 1.8 |
| [^18] | `outputs/report06_derived.json` | `slope.anchor_window_ghz[1]` | 18.2 |
| [^19] | `outputs/report06_derived.json` | `slope.anchor_band_matched_source` | Yuan theta=90 복원 실측곡선 (EuCAP 2025) |
| [^20] | `outputs/report06_derived.json` | `slope.anchor_band_matched_window_ghz[0]` | 1.8 |
| [^21] | `outputs/report06_derived.json` | `slope.anchor_band_matched_window_ghz[1]` | 6 |
| [^22] | `outputs/report06_derived.json` | `slope.campaign_window_ghz[0]` | 1.843 |
| [^23] | `outputs/report06_derived.json` | `slope.campaign_window_ghz[1]` | 5.21 |
| [^24] | `outputs/p3_validation_v2.json` | `our_operating_band` | (9항목 묶음) |
| [^25] | `outputs/report06_derived.json` | `slope.rows[0].gap_db` | 2.444 |
| [^26] | `outputs/report06_derived.json` | `slope.rows[0].gap_db_current_generation` | 0.07819 |
| [^27] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^28] | `outputs/angle_gamma_impact.json` | `_meta.generated` | 2026-08-07 10:58:22 |